In [17]:
import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('heart.csv')

In [3]:
df.sample(10)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
332,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
557,48,1,0,122,222,0,0,186,0,0.0,2,0,2,1
862,59,1,3,170,288,0,0,159,0,0.2,1,0,3,0
79,52,1,1,134,201,0,1,158,0,0.8,2,1,2,1
219,54,1,0,122,286,0,0,116,1,3.2,1,2,2,0
82,46,1,2,150,231,0,1,147,0,3.6,1,0,2,0
576,52,1,3,152,298,1,1,178,0,1.2,1,0,3,1
887,57,1,0,110,201,0,1,126,1,1.5,1,0,1,1
354,57,1,1,124,261,0,1,141,0,0.3,2,0,3,0
550,69,1,2,140,254,0,0,146,0,2.0,1,3,3,0


### One-Hot Encoding 
#### chest pain type (1–4)
#### rest ecg (0–2)
#### st slope (1–3)
#### thalassemia (3, 6, 7)

In [4]:
#### I will build NN from scratch and for that I must convert bool to float
df = pd.get_dummies(df,columns=['cp','restecg','slope','thal'],dtype=float)

In [5]:
df.sample(10)

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,target,...,restecg_0,restecg_1,restecg_2,slope_0,slope_1,slope_2,thal_0,thal_1,thal_2,thal_3
930,65,1,135,254,0,127,0,2.8,1,0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
452,66,0,178,228,1,165,1,1.0,2,0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
885,57,1,165,289,1,124,0,1.0,3,0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
775,51,1,100,222,0,143,1,1.2,0,1,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
374,46,0,142,177,0,160,1,1.4,0,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
606,66,1,112,212,0,132,1,0.1,1,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
268,58,1,132,224,0,173,0,3.2,2,0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
517,41,0,105,198,0,168,0,0.0,1,1,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
544,54,1,125,273,0,152,0,0.5,1,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
263,55,0,132,342,0,166,0,1.2,0,1,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


### Train-Test Split

In [9]:
X =  df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

#### Standart Scaling

In [14]:
continuous_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

In [15]:
# dataframes to numpy arrays
X_train = X_train.values.astype(float)
X_test = X_test.values.astype(float)
y_train = y_train.values.astype(float).reshape(-1, 1)
y_test = y_test.values.astype(float).reshape(-1, 1)

In [19]:
def initialize_parameters(n_x, n_h1=6 , n_h2=4 , n_y=1, seed=1):
    rng = np.random.default_rng(seed)
    W1 = rng.standard_normal((n_x, n_h1)) * np.sqrt(2 / n_x)
    b1 = np.zeros((1, n_h1))
    W2 = rng.standard_normal((n_h1, n_h2)) * np.sqrt(2 / n_h1)
    b2 = np.zeros((1, n_h2))
    W3 = rng.standard_normal((n_h2, n_y)) * np.sqrt(2 / n_h2)
    b3 = np.zeros((1, n_y))
    return {"W1": W1, "b1": b1, "W2": W2, "b2": b2, "W3": W3, "b3": b3}

### forward propagation

In [20]:
def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    Z = np.clip(Z, -500, 500) 
    return 1 / (1 + np.exp(-Z))

def forward_propagation(X, params):
    Z1 = X @ params["W1"] + params["b1"]
    A1 = relu(Z1)

    Z2 = A1 @ params["W2"] + params["b2"]
    A2 = relu(Z2)

    Z3 = A2 @ params["W3"] + params["b3"]
    A3 = sigmoid(Z3)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3, "X": X}
    return A3, cache

#### cost function

In [22]:
def compute_cost(A3, Y):
    m = Y.shape[0]
    eps = 1e-8 
    cost = -np.mean(Y * np.log(A3 + eps) + (1 - Y) * np.log(1 - A3 + eps))
    return cost

#### back propagation

In [21]:
def backward_propagation(Y, cache, params):
    m = Y.shape[0]
    X, A1, A2, A3 = cache["X"], cache["A1"], cache["A2"], cache["A3"]
    Z1, Z2 = cache["Z1"], cache["Z2"]

    dZ3 = A3 - Y
    dW3 = (A2.T @ dZ3) / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    dA2 = dZ3 @ params["W3"].T
    dZ2 = dA2 * relu_derivative(Z2)
    dW2 = (A1.T @ dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = dZ2 @ params["W2"].T
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = (X.T @ dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2, "dW3": dW3, "db3": db3}
    return grads

#### Update Parameters

In [23]:
def update_parameters(params, grads, learning_rate):
    params["W1"] -= learning_rate * grads["dW1"]
    params["b1"] -= learning_rate * grads["db1"]
    params["W2"] -= learning_rate * grads["dW2"]
    params["b2"] -= learning_rate * grads["db2"]
    params["W3"] -= learning_rate * grads["dW3"]
    params["b3"] -= learning_rate * grads["db3"]
    return params

In [24]:
def get_mini_batches(X, Y, batch_size, seed=None):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(X.shape[0])   # mix data every epoch
    X_shuffled, Y_shuffled = X[idx], Y[idx]

    batches = []
    for start in range(0, X.shape[0], batch_size):
        end = start + batch_size
        batches.append((X_shuffled[start:end], Y_shuffled[start:end]))
    return batches


def train_model(X_train, y_train, learning_rate=0.05, batch_size=16, epochs=500):
    n_x = X_train.shape[1]
    params = initialize_parameters(n_x)

    for epoch in range(epochs):
        batches = get_mini_batches(X_train, y_train, batch_size, seed=epoch)

        for X_batch, y_batch in batches:
            A3, cache = forward_propagation(X_batch, params)
            grads = backward_propagation(y_batch, cache, params)
            params = update_parameters(params, grads, learning_rate)

        # check cost after every epoch train data
        A3_full, _ = forward_propagation(X_train, params)
        cost = compute_cost(A3_full, y_train)
        if epoch % 50 == 0:
            print(f"Epoch {epoch}: cost = {cost:.4f}")

    return params

In [29]:
def train_model(X_train, y_train, X_val, y_val,
                 learning_rate=0.05, batch_size=16, epochs=1000, patience=30):
    n_x = X_train.shape[1]
    params = initialize_parameters(n_x)

    best_val_cost = np.inf
    best_params = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        batches = get_mini_batches(X_train, y_train, batch_size, seed=epoch)
        for X_batch, y_batch in batches:
            A3, cache = forward_propagation(X_batch, params)
            grads = backward_propagation(y_batch, cache, params)
            params = update_parameters(params, grads, learning_rate)

        A3_val, _ = forward_propagation(X_val, params)
        val_cost = compute_cost(A3_val, y_val)

        if val_cost < best_val_cost:
            best_val_cost = val_cost
            best_params = {k: v.copy() for k, v in params.items()}  # .copy() vacib!
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            print(f"Epoch {epoch}-də early stopping (ən yaxşı val_cost={best_val_cost:.4f})")
            break

    return best_params

In [30]:
### Prediction
def predict(X, params, threshold=0.5):
    A3, _ = forward_propagation(X, params)
    return (A3 >= threshold).astype(int)

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def confusion_matrix(y_true, y_pred):
    y_true = y_true.astype(int).flatten()
    y_pred = y_pred.astype(int).flatten()
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])

best_params = train_model(X_train, y_train, X_val, y_val)

y_pred_test = predict(X_test, best_params)
print("Test accuracy:", accuracy(y_test, y_pred_test))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_test))

NameError: name 'X_val' is not defined